In [1]:
import numpy as np
import pandas as pd

# Генерація даних для Варіанта 2 (Львів)
np.random.seed(2)
city = "Львів"
base_temp = 8.5
amplitude = 12.0
base_consumption = 2600.0
k = 40.0
noise_sd = 50.0

months = np.arange(1, 13)
temp = base_temp + amplitude * np.cos((months - 7) / 12 * 2 * np.pi)
temp = np.round(temp + np.random.normal(0, 0.5, size=12), 1)

consumption = base_consumption - k * (temp - base_temp) + np.random.normal(0, noise_sd, size=12)
consumption = np.round(consumption, 0)

city_data = pd.DataFrame({
    "місяць": months, 
    "температура": temp, 
    "споживання_МВтгод": consumption,
})

print("--- Спостережені дані міста Львів ---")
print(city_data)

# Завдання 2. Підбір лінійної моделі
b, a = np.polyfit(city_data["температура"], city_data["споживання_МВтгод"], 1)
print(f"\nПідібране рівняння: споживання ≈ {a:.1f} + ({b:.1f}) · температура")

city_data["передбачення"] = a + b * city_data["температура"]

--- Спостережені дані міста Львів ---
    місяць  температура  споживання_МВтгод
0        1         -3.7             3090.0
1        2         -1.9             2960.0
2        3          1.4             2911.0
3        4          9.3             2538.0
4        5         13.6             2395.0
5        6         18.5             2259.0
6        7         20.8             2071.0
7        8         18.3             2208.0
8        9         14.0             2336.0
9       10          8.0             2612.0
10      11          2.8             2841.0
11      12         -0.7             2919.0

Підібране рівняння: споживання ≈ 2923.7 + (-39.3) · температура


### Відповідь на Завдання 1. Класифікація сценарію
1. **Детермінована чи стохастична?** Ця модель є **стохастичною**. Хоча її базова математична структура ($a + b \cdot \text{температура}$) є детермінованою формулою, реальні дані містять випадковий випадковий залишок (шум `noise_sd`), тому модель описує процес як суму детермінованої закономірності та випадкового відхилення.
2. **Балансова, оптимізаційна, імітаційна чи жодна з них?** Ця модель належить до категорії **«жодна з цих трьох»**. Вона є описовою та прогнозною статистичною моделлю: у ній немає рівнянь збереження (як у балансових), пошуку оптимального значення (як в оптимізаційних) чи покрокової симуляції еволюції системи в часі (як в імітаційних).
3. **Статична чи динамічна?** Модель є **статичною**. Передбачення споживання для конкретного місяця залежить виключно від температури повітря у цьому ж місяці, а не від попередніх станів чи історії споживання за минулі місяці.

### Відповідь на Завдання 2. Інтерпретація нахилу $b$
* **Знак коефіцієнта $b$:** Отриманий коефіцієнт $b$ за своєю природою має бути **від'ємним**. Оскільки зниження температури спричиняє додаткові витрати енергії на опалення, зі зменшенням температури споживання зростає (і навпаки). Підібране на власних даних значення $b$ повністю узгоджується з цим фізичним та економічним сенсом (має мінусовий знак).

In [2]:
# Завдання 3. Обчислення залишків і коефіцієнта детермінації R²
residuals = city_data["споживання_МВтгод"] - city_data["передбачення"]
city_data["залишок"] = np.round(residuals, 1)

ss_res = (residuals ** 2).sum()
ss_tot = ((city_data["споживання_МВтгод"] - city_data["споживання_МВтгод"].mean()) ** 2).sum()
r_squared = 1 - (ss_res / ss_tot)

print("--- Таблиця результатів і залишків ---")
print(city_data[["місяць", "температура", "споживання_МВтгод", "передбачення", "залишок"]])
print(f"\nКоефіцієнт детермінації R² = {r_squared:.4f}")

--- Таблиця результатів і залишків ---
    місяць  температура  споживання_МВтгод  передбачення  залишок
0        1         -3.7             3090.0   3069.014688     21.0
1        2         -1.9             2960.0   2998.305314    -38.3
2        3          1.4             2911.0   2868.671463     42.3
3        4          9.3             2538.0   2558.335881    -20.3
4        5         13.6             2395.0   2389.419044      5.6
5        6         18.5             2259.0   2196.932417     62.1
6        7         20.8             2071.0   2106.581551    -35.6
7        8         18.3             2208.0   2204.789014      3.2
8        9         14.0             2336.0   2373.705850    -37.7
9       10          8.0             2612.0   2609.403761      2.6
10      11          2.8             2841.0   2813.675284     27.3
11      12         -0.7             2919.0   2951.165732    -32.2

Коефіцієнт детермінації R² = 0.9903


### Відповідь на Завдання 3. Оцінка адекватності моделі
* **Структура залишків:** При аналізі графіка чи таблиці залишків видно, що вони хаотично розкидані навколо нульової лінії без виражених систематичних трендів, дуг чи лійкоподібних розширень.
* **Висновок про адекватність:** Отримане високе значення $R^2$ у поєднанні з відсутністю прихованої нелінійної структури в залишках підтверджує, що підібрана лінійна модель є **адекватною** для прогнозування місячного споживання електроенергії в межах даного діапазону температур.

### Відповідь на Завдання 4. Верифікація і валідація
1. **Верифікація:** Для нашої моделі верифікація означає перевірку правильності реалізації алгоритму підбору та розрахунку. Конкретним прикладом є ручний підрахунок передбаченого значення за формулою $a + b \cdot \text{температура}$ для конкретного місяця (наприклад, січня) та порівняння цього результату з тим значенням, яке автоматично вивелося в колонці `передбачення` нашого DataFrame. Якщо значення збігаються, програмування моделі виконано без помилок.
2. **Валідація:** Валідація перевіряє, наскільки модель відповідає реальному світу й чи здатна вона адекватно працювати на нових даних. Для чесної валідації 12 спостережень, на яких підбиралися параметри $a$ і $b$, недостатньо (оскільки моделі «знайомі» ці точки). Повноцінна валідація вимагає залучення незалежних даних іншого року, які не брали участі в побудові рівняння регресії.

### Відповідь на Завдання 5. Пропозиція вдосконалення моделі
* **Обґрунтування вибору:** Оскільки наша лінійна модель продемонструвала високе значення $R^2$ і залишки, що випадково коливаються навколо нуля без систематичних трендів, модель слід визнати **адекватною** для поставленої мети.
* **Компроміс «точність–простота»:** Додаткове ускладнення моделі (наприклад, введення нелінійних членів чи додаткових складних факторів) тут є **непотрібним**. З огляду на принцип простоти (бритва Оккама), поточна лінійна модель забезпечує достатню точність прогнозу без надмірного ускладнення структури, що робить її оптимальною.

---

### Відповіді на контрольні питання

1. **Як розмежувати верифікацію та валідацію на прикладі власної моделі?**
   Верифікація перевіряє внутрішню коректність коду (наприклад, чи правильно підставилися коефіцієнти $a$ і $b$ у формулу розрахунку), тоді як валідація перевіряє зовнішню придатність моделі на нових, незалежних від набору навчання даних.

2. **За якими незалежними ознаками класифікується модель?**
   Модель класифікується за характером зв'язків (стохастична), за призначенням/ціллю (описова/прогнозна, не балансова і не оптимізаційна) та за врахуванням фактора часу (статична). Вони не суперечать одна одній, оскільки описують різні виміри системи.

3. **Що показує $R^2$ і чи доводить він придатність моделі для екстраполяції?**
   $R^2$ показує частку поясненої дисперсії залежної змінної в межах наявного діапазону. Високий $R^2$ **не доводить** придатності моделі для екстраполяції за межі спостережень, оскільки поза цим діапазоном лінійний зв'язок може кардинально порушитися.

4. **Чому структура залишків інформативніша за сумарну величину відхилень?**
   Аналіз структури залишків дозволяє побачити систематичні помилки (наприклад, якщо всі зимові місяці модель систематично недооцінює). Сумарна чисередня величина може згладжувати такі похибки через взаємне компенсаційне погашення додатних і від'ємних значень.